In [0]:
from pyspark.sql.functions import col, to_date, date_format, when, lit, to_timestamp, hour
from pyspark.sql.types import DateType

# --- START: Get secret password from ADF parameter ---
dbutils.widgets.text("sql_db_password", "", "SQL DB Password")
sql_db_password = dbutils.widgets.get("sql_db_password")
print(f"SQL Password received (first 3 chars): {sql_db_password[:3]}***")
# --- END ---

# 1. Define your exact CSV file path
csv_file_path = "dbfs:/FileStore/shared_uploads/sarathchandra1308@gmail.com/Data-1.csv"

# 2. Read the CSV file
df = spark.read.csv(csv_file_path, header=True, inferSchema=True)
print(f"Number of rows read from CSV: {df.count()}")
df.printSchema()
df.show(5, truncate=False)

# --- Data Transformation ---
df = df.withColumn("DATOP", to_date(col("DATOP"), "yyyy-MM-dd"))

df = df.withColumn("STD_Timestamp", to_timestamp(col("STD"), "MM-dd-yyyy HH:mm:ss")) \
       .withColumn("Flight_Day", date_format(col("STD_Timestamp"), "EEE"))

df = df.withColumn("Delay_Status",
                   when(col("target") <= 5, "On-time")
                   .when((col("target") > 5) & (col("target") <= 30), "Minor Delay")
                   .otherwise("Major Delay"))

df = df.withColumn("Time_Slot",
                   when((hour(col("STD_Timestamp")) >= 5) & (hour(col("STD_Timestamp")) < 12), "Morning")
                   .when((hour(col("STD_Timestamp")) >= 12) & (hour(col("STD_Timestamp")) < 17), "Afternoon")
                   .when((hour(col("STD_Timestamp")) >= 17) & (hour(col("STD_Timestamp")) < 21), "Evening")
                   .otherwise("Night"))

df = df.filter(col("STD").isNotNull() & col("STA").isNotNull() & col("target").isNotNull())

final_df = df.select(
    "ID", "FLTID", "DEPSTN", "ARRSTN", "STD", "STA", "STATUS",
    "AC", "target", "DATOP", "Delay_Status", "Flight_Day", "Time_Slot"
)

final_df.printSchema()
final_df.show(5, truncate=False)

# --- JDBC Write to Azure SQL ---
jdbc_url = "jdbc:sqlserver://sher29.database.windows.net:1433;database=sher1;encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"

connection_properties = {
    "user": "sarath",
    "password": sql_db_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

print(f"Number of rows in final_df before writing: {final_df.count()}")

try:
    final_df.write \
        .jdbc(url=jdbc_url, table="Flights", mode="overwrite", properties=connection_properties)
    print("\n--- Data successfully written to Azure SQL Database 'Flights' table ---")
except Exception as e:
    print(f"Error loading data into Flights table: {e}")
    raise e

print("\n--- Script execution completed ---")


SQL Password received (first 3 chars): !sK***
Number of rows read from CSV: 107833
root
 |-- ID: string (nullable = true)
 |-- DATOP: date (nullable = true)
 |-- FLTID: string (nullable = true)
 |-- DEPSTN: string (nullable = true)
 |-- ARRSTN: string (nullable = true)
 |-- STD: timestamp (nullable = true)
 |-- STA: string (nullable = true)
 |-- STATUS: string (nullable = true)
 |-- AC: string (nullable = true)
 |-- target: double (nullable = true)

+----------+----------+--------+------+------+-------------------+-------------------+------+---------+------+
|ID        |DATOP     |FLTID   |DEPSTN|ARRSTN|STD                |STA                |STATUS|AC       |target|
+----------+----------+--------+------+------+-------------------+-------------------+------+---------+------+
|train_id_0|2016-01-03|TU 0712 |CMN   |TUN   |2016-01-03 10:30:00|2016-01-03 12.55.00|ATA   |TU 32AIMN|260.0 |
|train_id_1|2016-01-13|TU 0757 |MXP   |TUN   |2016-01-13 15:05:00|2016-01-13 16.55.00|ATA   |TU 31BIMO

In [0]:
from pyspark.sql.functions import col, to_date, date_format, when, lit, to_timestamp, hour
from pyspark.sql.types import DateType


# 1. Define your exact CSV file path
csv_file_path = "dbfs:/FileStore/shared_uploads/sarathchandra1308@gmail.com/Data-1.csv"

# 2. Read the CSV file
df = spark.read.csv(csv_file_path, header=True, inferSchema=True)

print(f"Number of rows read from CSV: {df.count()}") # This will show the number of rows read from the CSV
print("\n--- Inferred Schema from CSV ---")
df.printSchema() # This will show the inferred schema
print("\n--- First few rows of raw DataFrame ---")
df.show(5, truncate=False) # This will show the first few rows

# --- Start of Data Pre-processing / Transformation ---

# Ensure DATOP is correctly converted to DateType
# IMPORTANT: Adjust "yyyy-MM-dd" to match the actual format of DATOP in your CSV
df = df.withColumn("DATOP", to_date(col("DATOP"), "yyyy-MM-dd"))

# Calculate Flight_Day from STD (Assuming STD is string or timestamp that can be parsed)
# Assuming STD is 'MM-DD-YYYY HH:MM:SS' or similar. Adjust format string as needed.
# Let's first ensure STD is a timestamp if it isn't already.
# If STD is already timestamp/date type, just use date_format(col("STD"), "EEE")
df = df.withColumn("STD_Timestamp", to_timestamp(col("STD"), "MM-dd-yyyy HH:mm:ss")) \
         .withColumn("Flight_Day", date_format(col("STD_Timestamp"), "EEE")) # EEE gives Mon, Tue, etc.

# Calculate Delay_Status based on 'target' (assuming 'target' is delay in minutes)
df = df.withColumn("Delay_Status",
                    when(col("target") <= 5, "On-time") # Assuming <=5 is on-time
                    .when((col("target") > 5) & (col("target") <= 30), "Minor Delay") # Assuming Minor Delay up to 30 mins
                    .otherwise("Major Delay")) # Greater than 30 mins is Major Delay

# Re-calculate Time_Slot (if needed, based on STA or STD, depending on your logic)
# This is an example, adjust according to your definition of Time_Slot
df = df.withColumn("Time_Slot",
                    when((hour(col("STD_Timestamp")) >= 5) & (hour(col("STD_Timestamp")) < 12), "Morning")
                    .when((hour(col("STD_Timestamp")) >= 12) & (hour(col("STD_Timestamp")) < 17), "Afternoon")
                    .when((hour(col("STD_Timestamp")) >= 17) & (hour(col("STD_Timestamp")) < 21), "Evening")
                    .otherwise("Night"))

# Remove rows with nulls in critical columns
df = df.filter(col("STD").isNotNull() & col("STA").isNotNull() & col("target").isNotNull())

# Select only the columns you want to write to your 'Flights' table in Azure SQL DB
# Ensure all your required columns are explicitly listed here.
final_df = df.select(
    "ID",
    "FLTID",
    "DEPSTN",
    "ARRSTN",
    "STD",
    "STA",
    "STATUS",
    "AC",
    "target",
    "DATOP",
    "Delay_Status",
    "Flight_Day",
    "Time_Slot"
)

# --- End of Data Pre-processing / Transformation ---
print("\n--- Final DataFrame Schema before writing to SQL ---")
final_df.printSchema()
print("\n--- First few rows of final DataFrame ---")
final_df.show(5, truncate=False)


# 3. Write the DataFrame to your Azure SQL Database
#    Ensure your JDBC connection details are correct.
jdbc_url = "jdbc:sqlserver://sher29.database.windows.net:1433;database=sher1;encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"

# Use the password retrieved from the widget (passed by ADF)
connection_properties = {
    "user": "sarath",
    "password":"!sKashya1", # <--- NOW USES THE PARAMETER
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

print(f"Number of rows in final_df before writing: {final_df.count()}")

try:
    final_df.write \
        .jdbc(url=jdbc_url, table="Flights", mode="overwrite", properties=connection_properties)
    print("\n--- Data successfully written to Azure SQL Database 'Flights' table ---")
except Exception as e:
    print(f"Error loading data into Flights table: {e}")
    raise e # Re-raise the exception so ADF marks the activity as failed

print("\n--- Script execution completed ---")

Number of rows read from CSV: 107833

--- Inferred Schema from CSV ---
root
 |-- ID: string (nullable = true)
 |-- DATOP: date (nullable = true)
 |-- FLTID: string (nullable = true)
 |-- DEPSTN: string (nullable = true)
 |-- ARRSTN: string (nullable = true)
 |-- STD: timestamp (nullable = true)
 |-- STA: string (nullable = true)
 |-- STATUS: string (nullable = true)
 |-- AC: string (nullable = true)
 |-- target: double (nullable = true)


--- First few rows of raw DataFrame ---
+----------+----------+--------+------+------+-------------------+-------------------+------+---------+------+
|ID        |DATOP     |FLTID   |DEPSTN|ARRSTN|STD                |STA                |STATUS|AC       |target|
+----------+----------+--------+------+------+-------------------+-------------------+------+---------+------+
|train_id_0|2016-01-03|TU 0712 |CMN   |TUN   |2016-01-03 10:30:00|2016-01-03 12.55.00|ATA   |TU 32AIMN|260.0 |
|train_id_1|2016-01-13|TU 0757 |MXP   |TUN   |2016-01-13 15:05:00|2016-01

In [0]:
from pyspark.sql.functions import avg, count, concat_ws, substring, lit, sum as spark_sum, round, when, col # Ensure 'when' and 'col' are imported if not already

print("\nPerforming aggregations for 'Aggregated_Delays' table...")

# Derive 'Airline' from FLTID and 'Route'
# --- CHANGE HERE: use final_df instead of final_flights_df ---
agg_df = final_df.withColumn("Airline", substring(col("FLTID"), 1, 2)) \
                         .withColumn("Route", concat_ws("-", col("DEPSTN"), col("ARRSTN")))

# Calculate aggregations
aggregated_delays_df = agg_df.groupBy("Airline", "Flight_Day", "Time_Slot", "Route") \
    .agg(
        avg("target").alias("Avg_Delay"),
        count("*").alias("Total_Flight_Count"),
        spark_sum(when((col("Delay_Status") == "Minor Delay") | (col("Delay_Status") == "Major Delay"), 1).otherwise(0)).alias("Delayed_Flight_Count")
    ) \
    .withColumn(
        "Delay_Percentage",
        round((col("Delayed_Flight_Count").cast("float") / col("Total_Flight_Count")) * 100, 2)
    ) \
    .select(
        "Airline",
        col("Avg_Delay").cast("float"), # Ensure float type for SQL
        col("Total_Flight_Count").alias("Flight_Count").cast("int"),
        col("Delay_Percentage").cast("float"),
        "Flight_Day",
        "Time_Slot",
        "Route"
    )

print("Aggregated_Delays DataFrame schema:")
aggregated_delays_df.printSchema()
print("First 5 rows of Aggregated_Delays:")
aggregated_delays_df.show(5, truncate=False)

try:
    # --- CHANGE HERE: Use .jdbc method with connection_properties ---
    aggregated_delays_df.write \
        .jdbc(url=jdbc_url, table="Aggregated_Delays", mode="overwrite", properties=connection_properties)
    print("Data successfully loaded into 'Aggregated_Delays' table.")
except Exception as e:
    print(f"Error loading data into Aggregated_Delays table: {e}")

print("\nAll data loading to Azure SQL complete!")


Performing aggregations for 'Aggregated_Delays' table...
Aggregated_Delays DataFrame schema:
root
 |-- Airline: string (nullable = true)
 |-- Avg_Delay: float (nullable = true)
 |-- Flight_Count: integer (nullable = false)
 |-- Delay_Percentage: float (nullable = true)
 |-- Flight_Day: string (nullable = true)
 |-- Time_Slot: string (nullable = false)
 |-- Route: string (nullable = false)

First 5 rows of Aggregated_Delays:
+-------+---------+------------+----------------+----------+---------+-------+
|Airline|Avg_Delay|Flight_Count|Delay_Percentage|Flight_Day|Time_Slot|Route  |
+-------+---------+------------+----------------+----------+---------+-------+
|TU     |33.84127 |126         |71.43           |Tue       |Morning  |TUN-VIE|
|TU     |84.78261 |23          |86.96           |Sat       |Afternoon|TUN-SFA|
|AT     |6.0      |10          |50.0            |Wed       |Morning  |OUD-MRS|
|TU     |352.5    |4           |75.0            |Sat       |Night    |MIR-CDG|
|TU     |41.18919 

In [0]:
# --- Preparing and loading data into Azure SQL 'Aircraft' table ---
print("\nPreparing and loading data into Azure SQL 'Aircraft' table...")

# Assuming FLTID and AC form a unique pair for an aircraft entry
# IMPORTANT: Change 'final_flights_df' to 'final_df' here
aircraft_df = final_df.select("FLTID", "AC").distinct()

try:
    aircraft_df.write \
        .jdbc(url=jdbc_url, table="Aircraft", mode="overwrite", properties=connection_properties) # Use .jdbc with predefined url and properties
    print("Data successfully loaded into 'Aircraft' table.")
except Exception as e:
    print(f"Error loading data into Aircraft table: {e}")
    raise e # Re-raise the exception to ensure the ADF pipeline fails if this step fails

# print("\nAll data loading to Azure SQL complete!") # This might be at the very end of your full notebook


Preparing and loading data into Azure SQL 'Aircraft' table...
Data successfully loaded into 'Aircraft' table.


In [0]:
print("\nPreparing and loading data into Azure SQL 'Routes' table...")

# IMPORTANT: Change 'final_flights_df' to 'final_df' here
routes_df = final_df.select("FLTID", "DEPSTN", "ARRSTN").distinct()

try:
    # IMPORTANT: Use .jdbc method with jdbc_url and connection_properties
    routes_df.write \
        .jdbc(url=jdbc_url, table="Routes", mode="overwrite", properties=connection_properties)
    print("Data successfully loaded into 'Routes' table.")
except Exception as e:
    print(f"Error loading data into Routes table: {e}")
    raise e # Re-raise the exception to ensure the ADF pipeline fails if this step fails


Preparing and loading data into Azure SQL 'Routes' table...
Data successfully loaded into 'Routes' table.
